In [ ]:
"""
discretize 4 numeric XI-based features into 3 bins
learn with multinomical logistic regresson
team strength as latent variable
inference with VE
46% accuracy
"""

from sklearn.metrics import classification_report
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

__BINS = 5
# datast
data = pd.read_csv('../../cleaned_data/final_dataset.csv')
feature_data = pd.read_csv('../../dataset/match_features_merged.csv', usecols=[
                           'game_id', 'season_ppg_home', 'season_ppg_away', 'season_ppg_diff'])
data = data.merge(feature_data, on='game_id', how='inner')
print(data.head())
# outcome as numerical category
data['result_code'] = data['result'].map(
    {'Home Win': 0, 'Draw': 1, 'Away Win': 2})
data['MatchOutcome'] = data['result_code']

# discretize continuous features. binning.
features_home = ['home_avg_market_value',
                 'home_nationalities', 'home_avg_age', 'home_total_minutes']
features_away = ['away_avg_market_value',
                 'away_nationalities', 'away_avg_age', 'away_total_minutes']

kbins_home = KBinsDiscretizer(
    n_bins=__BINS, encode='ordinal', strategy='uniform')
kbins_away = KBinsDiscretizer(
    n_bins=__BINS, encode='ordinal', strategy='uniform')

data_discrete_home = kbins_home.fit_transform(data[features_home]).astype(int)
data_discrete_away = kbins_away.fit_transform(data[features_away]).astype(int)

# put back discretized values back to data
for i, col in enumerate(features_home):
    data[f'disc_{col}'] = data_discrete_home[:, i]

for i, col in enumerate(features_away):
    data[f'disc_{col}'] = data_discrete_away[:, i]

# feature weight learning
X_home = data[
    [f'disc_{col}' for col in features_home]
]
print(X_home)
X_away = data[
    [f'disc_{col}' for col in features_away]
]
y = data['MatchOutcome']

# split 8/2
X_home_train, X_home_test, X_away_train, X_away_test, y_train, y_test = train_test_split(
    X_home, X_away, y, test_size=0.2, random_state=42)


# learn feature weights with logistic regression
clf_home = LogisticRegression(
    multi_class='multinomial', solver='lbfgs', max_iter=500)
clf_home.fit(X_home_train, y_train)

clf_away = LogisticRegression(
    multi_class='multinomial', solver='lbfgs', max_iter=500)
clf_away.fit(X_away_train, y_train)

# learned weights
weights_home = clf_home.coef_.mean(axis=0)
weights_away = clf_away.coef_.mean(axis=0)

# display weights
weights_df = pd.DataFrame({
    'Feature': features_home + features_away,
    'Weights': list(weights_home) + list(weights_away)
})
print("Learned Feature Weights:\n", weights_df)


# apply weights
data['HomeStrength'] = (X_home*weights_home).sum(axis=1)
data['AwayStrength'] = (X_away*weights_away).sum(axis=1)

# explicitly re-discretize to ensure clear discrete bins
kbins_home_strength = KBinsDiscretizer(
    n_bins=__BINS, encode='ordinal', strategy='uniform')
data['HomeStrength'] = kbins_home_strength.fit_transform(
    data[['HomeStrength']]).astype(int).flatten()

kbins_away_strength = KBinsDiscretizer(
    n_bins=__BINS, encode='ordinal', strategy='uniform')
data['AwayStrength'] = kbins_away_strength.fit_transform(
    data[['AwayStrength']]).astype(int).flatten()

# setup network
"""
disc_home_avg_market_value ──┐
disc_home_nationalities ─────┤
disc_home_avg_age ───────────┤ → HomeStrength ────┐
disc_home_total_minutes ─────┘                    │
                                                  │ → MatchOutcome
disc_away_avg_market_value ──┐                    │
disc_away_nationalities ─────┤                    │
disc_away_avg_age ───────────┤ → AwayStrength ────┘
disc_away_total_minutes ─────┘
"""
structure = [
    ('disc_home_avg_market_value', 'HomeStrength'),
    ('disc_home_nationalities', 'HomeStrength'),
    ('disc_home_avg_age', 'HomeStrength'),
    ('disc_home_total_minutes', 'HomeStrength'),

    ('disc_away_avg_market_value', 'AwayStrength'),
    ('disc_away_nationalities', 'AwayStrength'),
    ('disc_away_avg_age', 'AwayStrength'),
    ('disc_away_total_minutes', 'AwayStrength'),

    ('HomeStrength', 'MatchOutcome'),
    ('AwayStrength', 'MatchOutcome')
]

# Define explicit Bayesian network explicitly using new class
model = DiscreteBayesianNetwork(structure)

# estimate CPTs. BDeu: Bayesian Dirichlet equivalent uniform. avoid zeros.
model.fit(data, estimator=BayesianEstimator, prior_type='BDeu')

# Explicit inference example
infer = VariableElimination(model)


# print(query_result)
# predict explicitly using Bayesian network inference on test data
# Compute test strengths explicitly
data_test = X_home_test.copy()
data_test['HomeStrength'] = (X_home_test * weights_home).sum(axis=1)
data_test['AwayStrength'] = (X_away_test * weights_away).sum(axis=1)

# discretize strengths explicitly (using previously fitted discretizers!)
data_test['HomeStrength'] = kbins_home_strength.transform(
    data_test[['HomeStrength']]).astype(int).flatten()
data_test['AwayStrength'] = kbins_away_strength.transform(
    data_test[['AwayStrength']]).astype(int).flatten()
predictions = []
for idx, row in data_test.iterrows():
    evidence = {
        'disc_home_avg_market_value': row['disc_home_avg_market_value'],
        'disc_home_nationalities': row['disc_home_nationalities'],
        'disc_home_avg_age': row['disc_home_avg_age'],
        'disc_home_total_minutes': row['disc_home_total_minutes'],
        'disc_away_avg_market_value': X_away_test.loc[idx]['disc_away_avg_market_value'],
        'disc_away_nationalities': X_away_test.loc[idx]['disc_away_nationalities'],
        'disc_away_avg_age': X_away_test.loc[idx]['disc_away_avg_age'],
        'disc_away_total_minutes': X_away_test.loc[idx]['disc_away_total_minutes'],
        'HomeStrength': row['HomeStrength'],
        'AwayStrength': row['AwayStrength']
    }
    result = infer.query(['MatchOutcome'], evidence=evidence)
    pred = result.values.argmax()
    predictions.append(pred)

# evaluate explicitly accuracy
print(classification_report(y_test, predictions))

   game_id        date  home_club_id  away_club_id  home_club_goals  \
0  2321027  2013-08-11          33.0          41.0              3.0   
1  2321033  2013-08-10          23.0          86.0              0.0   
2  2321044  2013-08-18          16.0          23.0              2.0   
3  2321060  2013-08-25          23.0          24.0              0.0   
4  2321072  2013-09-14          16.0          41.0              6.0   

   away_club_goals  home_avg_market_value_x  home_nationalities  home_avg_age  \
0              3.0             3.217308e+06                 9.0     25.897120   
1              1.0             2.696429e+05                 6.0     27.092929   
2              1.0             3.619643e+06                 6.0     24.926762   
3              2.0             3.125000e+05                 8.0     26.142263   
4              2.0             3.139286e+06                 6.0     24.916984   

   home_total_minutes  ...  star_index_away  star_index_diff  \
0               990.0 

KeyError: "['home_avg_market_value'] not in index"